# Summary and interpretation of saved modelling runs

Choose **one or several folders** below: a collection of runs, a single run, or an individual `optimization/SEED/runhistory.json`. The notebook reads saved histories directly, including runs without a final `trials.csv`.

It summarises coverage and execution failures, identifies best final-stage models, displays individual observation contributions, compares competitive parameter families and physical structures, and inspects saved predictions. A common pool of fully scored models supports analysis of adding or removing observation groups.

**Scientific interpretation:** the returned SMAC cost combines reduced chi-square and empirical image losses; it is not a joint reduced chi-square or a posterior probability. Unscored observables remain missing. Adaptive optimisation samples and their ranges are not posterior distributions or credible intervals. The notebook never launches MCFOST, changes weights, modifies observations, or rescales recorded costs.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
from IPython.display import display, Markdown

# Locate the helper when starting Jupyter from the notebook, repository or workspace.
_candidates = [Path.cwd()]
for _base in [Path.cwd(), *Path.cwd().parents]:
    _candidates.extend([_base / 'scripts/post-processing',
                        _base / 'modelling_optimisation/scripts/post-processing'])
NOTEBOOK_DIR = next((p for p in _candidates if (p / 'run_summary_utils.py').is_file()), None)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Open this notebook from the repository or keep run_summary_utils.py beside it.')
sys.path.insert(0, str(NOTEBOOK_DIR))
from run_summary_utils import (
    SCORE_COLUMNS, SUBSETS, INTERFEROMETRY, PDI, load_runs, coverage_table,
    select_solutions, parameter_summary, stage_correlations, common_pool_comparison,
    prediction_tables, resolve_model_files, plot_costs, plot_convergence,
    plot_parameters, plot_amplitudes, plot_saved_profiles, plot_tradeoffs,
    parameter_pair_analysis, select_parameter_pairs, plot_selected_parameter_pairs,
    correlation_strength, correlation_metric, plot_parameter_correlation_heatmap,
)
# Inferred workspace above the repository; used to construct the default RUN_FOLDERS paths.
WORKSPACE_ROOT = NOTEBOOK_DIR.parents[2]
# Notebook table display width only; CSV exports retain all columns.
pd.set_option('display.max_columns', 25)
# Inline figure resolution and default grid visibility; saved PNGs use the export cell DPI.
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': False})
print('Python:', sys.executable)

## 1. Select folders and analysis options

Labels identify independent collections. Discovery is bounded to four levels of subfolders and skips the large trial-output trees. Overlapping inputs are deduplicated. Every optimiser seed remains a separate history.

`TOP_N` selects distinct sampled parameter configurations. Competitive models satisfy `cost <= best + max(ABSOLUTE_COST_ALLOWANCE, abs(best) * RELATIVE_COST_ALLOWANCE)` within each history and final-stage objective. This descriptive threshold does not imply a statistical confidence level.

In [ ]:
# Input collections: map a readable label to a parent folder, single run, seed folder, or runhistory.json.
# Uncomment/add entries to compare collections. Overlapping histories are loaded once; seeds stay separate.
RUN_FOLDERS = {
    'Dubrulle': WORKSPACE_ROOT / 'smac/1_akke_2zones_wider_tapered/dubrulle',
    # 'Earlier tapered': WORKSPACE_ROOT / 'smac/1_tapered_2zones_dubrulle',
    # 'Single run': WORKSPACE_ROOT / 'smac/1_akke_2zones_wider_tapered/dubrulle/sed_alma',
}
# Exclude recorded costs >= this value, including execution penalties such as 1e99.
# Success status, finite cost and absence of recorded failure are also required.
PENALTY_LIMIT = 1e90
# Maximum distinct final-stage configurations retained per history/objective in the top-model table.
# This does not limit the competitive set, correlation sample, or number of optimisation trials.
TOP_N = 200
# Fractional competitive-cost allowance above the best recorded cost: 0.10 = 10%; 1.0 = 100%.
# Threshold = best + max(abs(best) * this allowance, ABSOLUTE_COST_ALLOWANCE).
RELATIVE_COST_ALLOWANCE = 1.0
# Alternative competitive allowance in composite-objective units (not a percentage).
# The larger of the relative and absolute allowances is used; they are not added.
ABSOLUTE_COST_ALLOWANCE = 0.0

# None includes every independently sampled parameter in the saved ConfigSpaces.
# Supply a list, e.g. ['disk_total_dust_mass', 'disk_Rmid'], to focus the plots.
# Parameter names for comparison plots and selected parameter tables; None infers all fitted names.
# Pair screening still audits all independently varied parameters regardless of this list.
PARAMETERS = None
# Use logarithmic parameter axes for these names when all plotted values are positive.
# This changes display coordinates only, not costs, selection or rank-correlation coefficients.
LOG_PARAMETERS = {'disk_total_dust_mass', 'alpha_viscosity', 'zone_1_species_1_amax'}
# Quantitative within-history pair screening (descriptive thresholds, not significance tests).
# Sample used for correlation heatmaps and pair selection: all_final = all usable final-stage models;
# competitive = only models passing the competitive-cost threshold. Both coefficients are tabulated.
PAIR_SCOPE = 'all_final'  # 'competitive' or 'all_final'; both correlations are always reported.
# Rank-correlation metrics to display for every run; both are calculated on the same finite model pairs.
# Scatter panels keep the existing Spearman selection so the two metrics use identical axes and samples.
PAIR_METRICS = ('spearman', 'kendall_tau_b')
# Minimum number of distinct configurations with finite values on both axes for an eligible pair.
# Applied separately within each history/objective and sample scope; smaller samples appear as NA.
PAIR_MIN_MODELS = 20
# Minimum number of distinct numeric values required on EACH parameter axis.
# Constant or sparsely sampled axes fail this screen even if many models were evaluated.
PAIR_MIN_UNIQUE_VALUES = 3
# Minimum absolute Spearman coefficient for selecting association scatter plots (range 0 to 1).
# Heatmaps show every eligible coefficient. Lower to 0.2 to include weak pairs in scatter plots.
# Spearman selects the shared panels; Kendall Tau-b is shown for those same parameter pairs.
PAIR_MIN_ABS_RHO = 0.5
# Legacy option: the current per-run plotting section does not read this value.
# Use PAIR_MAX_PER_HISTORY and PAIR_PANELS_PER_FIGURE to control association plots.
PAIR_MAX_PLOTS = 60
# Maximum strongest eligible association pairs plotted for EACH history/objective (positive integer).
# There is no shared cap across modelling runs; this limit is applied after correlation screening.
PAIR_MAX_PER_HISTORY = 60
# Maximum scatter panels on one figure page (positive integer).
# Larger per-run selections continue on additional pages belonging to the same run.
PAIR_PANELS_PER_FIGURE = 6  # Paginate large selections within each run.
# Parameters with circular/periodic coordinates, excluded from both ordinary rank-correlation calculations.
# They are marked NA in heatmaps; this notebook does not compute circular correlations.
PAIR_PERIODIC_PARAMETERS = {'disk_pa'}
# True allows automatic scatter selection of pairs directly linked by ConfigSpace inequalities.
# Heatmaps always mark such pairs with a dagger/purple outline; imposed links can induce correlation.
PAIR_INCLUDE_CONSTRAINED = False
# Optional exact history labels to include in plots; None shows every history.
# Optional list of exact inventory history labels to include in plots and parameter-pair analysis.
# None includes all histories. Summary tables and common-pool calculations still use all loaded runs.
PLOT_HISTORIES = None
# Maximum number of plotted best-model rows used in saved PDI/ALMA radial-profile overlays.
# Uses their existing order; this does not limit other best-model or physical-structure plots.
PROFILE_MAX_MODELS = 6
# True displays original saved SED, visibility, PDI and ALMA diagnostic PNGs in the gallery.
# Reads existing images only; it does not recompute observables or launch MCFOST.
SHOW_SAVED_DIAGNOSTICS = False
# Maximum matched best models shown when the diagnostic gallery is enabled.
# Display priority is most scored observation groups, then lowest cost; this is not a joint ranking.
GALLERY_MAX_MODELS = 2
# Set this to one of the printed pool_group values when multiple compatible pools exist.
# Compatibility-group ID to inspect in the common-model-pool analysis (use a printed pool_group).
# None chooses the group with the largest number of saved compatible evaluations.
POOL_GROUP = None
# Ordered subset objectives used to quantify adding observations on the SAME saved model pool.
# Names must occur in SUBSETS; each step must retain all score terms from the previous step.
# No new simulations or reweighting are performed.
OBSERVATION_SEQUENCE = ['SED', 'SED+interferometry', 'All']

# True writes analysis CSV tables, PNG figures and JSON settings into OUTPUT_DIR.
# False keeps results in notebook memory; simulations and raw observations are never modified.
SAVE_OUTPUTS = False
if SAVE_OUTPUTS:
    if len(RUN_FOLDERS)>1:
        # Export destination: this branch derives it from the first selected input folder.
        # Set an explicit writable folder here if using a runhistory.json file as input.
        # It is created only when SAVE_OUTPUTS=True; exports replace existing files with the same names.
        OUTPUT_DIR = list(RUN_FOLDERS.values())[0].resolve().parent / 'run_summary_outputs'
    else:
        # Export destination: this branch derives it from the first selected input folder.
        # Set an explicit writable folder here if using a runhistory.json file as input.
        # It is created only when SAVE_OUTPUTS=True; exports replace existing files with the same names.
        OUTPUT_DIR = list(RUN_FOLDERS.values())[0] / 'run_summary_outputs'

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Internal figure registry used for previews and optional PNG exports; leave this empty initially.
figures = {}
# Internal display helper: retain a figure for export, display it, then close its live window.
def show_figure(name, figure):
    if figure is not None:
        figures[name] = figure
        display(figure)
        plt.close(figure)


## 2. Inventory, provenance and usable evaluations

A usable evaluation must have success status, a finite cost below the penalty threshold, a configuration, and no recorded failure. This excludes execution failures returned as `1e99` even when SMAC labels them successful.

The inventory records saved scenario seeds, the current physical template and ConfigSpace fingerprints, photon settings and stage manifests. Templates and launch scripts are current files, not immutable historical snapshots; identical hashes do not prove that historical observations and code were identical. The saved per-trial PDI correction flags are stronger evidence than the current launch script.

`evaluation_wall_hours` is the sum of recorded trial durations, not elapsed time or CPU-core hours.

In [ ]:
evaluations, inventory, records, configspaces = load_runs(RUN_FOLDERS, PENALTY_LIMIT)
coverage = coverage_table(evaluations)
if PARAMETERS is None:
    PARAMETERS = sorted({p['name'] for space in configspaces.values()
                         for p in space.get('hyperparameters', []) if p.get('type') != 'constant'})
    if not PARAMETERS:
        PARAMETERS = [c.removeprefix('param__') for c in evaluations if c.startswith('param__')
                      and pd.to_numeric(evaluations[c], errors='coerce').nunique() > 1]
display(inventory[['history', 'configured_max_budget', 'recorded_trials', 'requested_trials',
                   'launch_requests_correction', 'photon_settings']])
display(coverage)
display(inventory[['history', 'configspace_hash', 'template_hash', 'history_sha256']])
print(f'{len(inventory)} histories; {len(evaluations):,} recorded evaluations; '
      f'{evaluations.usable.sum():,} usable evaluations.')
exclusions = evaluations.loc[~evaluations.usable].groupby(['history', 'exclusion']).size().rename('count').reset_index()
display(exclusions)
# Separate real execution failures from observationally poor models.
failure_examples = evaluations.loc[evaluations.failure_reason.ne(''), ['history', 'config_id', 'budget', 'failure_reason']]
display(failure_examples.head(12))


### Cost bookkeeping: individual contributions versus recorded cost

The current objective is the sum of SED reduced chi-square, four instrument reduced chi-squares, PDI V/I/H losses and ALMA loss, restricted to the products requested for that evaluation. PDI and ALMA losses are not reduced chi-squares, so the composite total is not a reduced chi-square or a calibrated likelihood. Contributions may already contain internal weights; this audit never changes them. SSIM and log-likelihood diagnostics are not added.

`cost_difference = reconstructed_cost - cost`; a positive value means the saved terms sum to more than the recorded cost. Match tolerance is `abs(difference) <= 1e-8 + 1e-8 * abs(cost)`. Relative difference uses `abs(cost)` and is undefined at zero. Missing/non-finite required terms, unknown products or unexpected finite terms make reconstruction incomplete, even if the available sum happens to equal cost. Inactive observations remain missing, not zero. Products inferred from old payloads cannot prove that every historical contribution was saved.

The summary includes excluded trials separately: execution penalties are not expected to equal observation losses. The scatter uses only usable, complete records; mismatches remain visible. A matching sum verifies bookkeeping, not the scientific validity of the loss or historical code/data identity.


In [ ]:
audit_columns = ['evaluation_id', 'history', 'config_id', 'budget', 'stage', 'products',
                 'usable', 'exclusion', 'products_source', 'cost_audit_status', 'cost',
                 'available_score_sum', 'reconstructed_cost', 'cost_difference',
                 'cost_absolute_difference', 'cost_relative_difference',
                 'missing_score_terms', 'unknown_products', 'unexpected_score_terms',
                 *SCORE_COLUMNS.values()]
score_audit = evaluations.reindex(columns=audit_columns).copy()
score_audit_summary = (score_audit.groupby(
    ['history', 'budget', 'stage', 'products', 'usable', 'cost_audit_status'], dropna=False)
    .agg(evaluations=('evaluation_id', 'size'),
         max_absolute_difference=('cost_absolute_difference', 'max'),
         max_relative_difference=('cost_relative_difference', lambda x: x.abs().max()))
    .reset_index())
display(score_audit_summary)
display(score_audit.loc[score_audit.usable].head(20))
score_audit_issues = score_audit.loc[score_audit.usable & score_audit.cost_audit_status.ne('match')]
if not score_audit_issues.empty:
    display(Markdown('**Usable evaluations with incomplete or mismatching score records:**'))
    display(score_audit_issues.head(30))
audit_plot = evaluations.loc[evaluations.usable & evaluations.cost_terms_complete]
if not audit_plot.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for history, group in audit_plot.groupby('history', sort=False):
        axes[0].scatter(group.cost, group.reconstructed_cost, s=12, alpha=.5, label=history)
        axes[1].scatter(group.cost, group.cost_difference, s=12, alpha=.5)
    lo = min(audit_plot.cost.min(), audit_plot.reconstructed_cost.min())
    hi = max(audit_plot.cost.max(), audit_plot.reconstructed_cost.max())
    axes[0].plot([lo, hi], [lo, hi], 'k--', linewidth=1)
    axes[0].set(xlabel='Recorded cost', ylabel='Sum of saved objective contributions')
    axes[0].legend(fontsize=6)
    axes[1].axhline(0, color='k', linestyle='--', linewidth=1)
    axes[1].set(xlabel='Recorded cost', ylabel='Sum minus recorded cost')
    fig.tight_layout()
    show_figure('score_sum_vs_cost', fig)


## 3. Best models and observation contributions

Select the **configured final budget**, rather than falling back to an earlier stage when no final-stage evaluation succeeded. Budget numbers are local to a run: `1` may mean SED-only in one folder and all observations in another. The products column defines the observational objective.

The tables keep the lowest recorded score for each distinct configuration. They are best observed evaluations, not repeated-seed means or a claim of convergence. Repeated trials remain available in `evaluations`. Final `smac_results` reruns are not substituted for runhistory scores.

In [ ]:
best, top_models, competitive = select_solutions(
    evaluations, TOP_N, RELATIVE_COST_ALLOWANCE, ABSOLUTE_COST_ALLOWANCE)
missing_final = sorted(set(inventory.history) - set(best.history))
if missing_final:
    print('No usable configured-final-stage model:', *missing_final, sep='\n')
score_columns = ['history', 'config_id', 'budget', 'products', 'cost', *SCORE_COLUMNS.values()]
display(best.reindex(columns=score_columns))
parameter_columns = ['param__' + p for p in PARAMETERS if 'param__' + p in best]
display(best.reindex(columns=['history', 'config_id', 'cost', 'sed_ebminv', *parameter_columns]))
if PLOT_HISTORIES is None:
    plotted_best, plotted_competitive = best, competitive
else:
    plotted_best = best[best.history.isin(PLOT_HISTORIES)]
    plotted_competitive = competitive[competitive.history.isin(PLOT_HISTORIES)]
show_figure('observation_contributions', plot_costs(plotted_best))

## 4. Search progress and sensitivity to observation staging

Each curve tracks its own stage objective. A cost discontinuity when observations are added is not a convergence failure. Do not compare the numerical heights of curves for different product sets as if they were the same objective.

Rank correlations use only configurations evaluated at both neighbouring budgets within a history. Pairing uses parameter values, simulation seed and instance. These selected pairs can be biased by promotion. Repeated evaluations of the same pairing key are averaged only for this diagnostic.

In [ ]:
plot_evaluations = evaluations if PLOT_HISTORIES is None else evaluations[evaluations.history.isin(PLOT_HISTORIES)]
show_figure('convergence_by_stage', plot_convergence(plot_evaluations))
rank_correlations = stage_correlations(evaluations)
if rank_correlations.empty:
    print('No multi-budget histories to compare.')
else:
    display(rank_correlations)

## 5. Competitive physical solutions and parameter boundaries

Points show distinct competitive sampled configurations; stars identify the best observed model. No inverse-cost or likelihood weighting is applied. Narrow sampled ranges may reflect search concentration, fixed assumptions, selection or parameter bounds.

The boundary table measures the fraction of competitive models within 2% of each numerical interval (in log coordinates for log-sampled parameters). Categorical endpoints are not treated as continuous bounds. Inclination near zero makes disk PA difficult to interpret; PA also has an angular wrap at 0/180 degrees.

In [ ]:
parameter_ranges = parameter_summary(competitive, configspaces)
display(competitive.groupby('history').size().rename('competitive_distinct_models').to_frame())
show_figure('competitive_parameters', plot_parameters(
    plotted_competitive, plotted_best, PARAMETERS, LOG_PARAMETERS))
if not parameter_ranges.empty:
    boundary_flags = parameter_ranges.loc[
        parameter_ranges.fraction_near_lower_bound.gt(0) | parameter_ranges.fraction_near_upper_bound.gt(0)]
    display(boundary_flags)
else:
    boundary_flags = pd.DataFrame()
    print('No numeric competitive parameter summaries available.')

### Quantify parameter pairs: Spearman and Kendall Tau-b for each modelling run

Audit **every independently varied parameter pair**, separately within each modelling history (folder plus optimiser seed) and final-stage observation objective. Different histories, seeds and objectives are never pooled. Fixed/tied parameters are excluded. Nonnumeric categories are not assigned arbitrary numeric codes; periodic `disk_pa` requires circular analysis and is marked unavailable for both metrics.

**Calculate both coefficients on exactly the same finite paired model values:**

- **Spearman ρ** correlates the ranks, using average ranks for ties.
- **Kendall τ-b** compares concordant and discordant pairs, with normalisation accounting for ties on either parameter axis. The calculation explicitly uses `scipy.stats.kendalltau(..., variant='b')`.

The paired model counts, exclusions, deduplication and sample scopes are identical. The table retains `rho_competitive`, `tau_b_competitive`, `rho_all_final` and `tau_b_all_final`. No p-values are used for ranking or displayed as evidence of significance.

**First: a separate heatmap for each metric and each run.** Both heatmaps have the same parameter order, support masks, observation objective and model sample. Every eligible coefficient is annotated, including weak values below the scatter-selection threshold. Positive correlations are red, negative correlations blue; darker colours indicate larger absolute coefficients. Grey `NA` cells indicate insufficient models, insufficient variation, periodic parameters or undefined coefficients. They are not zero correlations. Diagonal self-correlations are omitted. Direct ConfigSpace relationships carry a purple outline and a dagger.

| Absolute coefficient (display bands) | Descriptive level |
|---|---|
| < 0.2 | Negligible |
| 0.2 to < 0.4 | Weak |
| 0.4 to < 0.6 | Moderate |
| 0.6 to < 0.8 | Strong |
| 0.8 to 1 | Very strong |

The same numeric bands make the two displays consistent; **they do not make Spearman and Kendall coefficients interchangeable**, nor establish equivalent significance or evidence.

**Then: matching association figures for each metric and each run.** Preserve the existing Spearman-based ranking and `PAIR_MIN_ABS_RHO` threshold, selecting up to `PAIR_MAX_PER_HISTORY` shared pairs per run. For every selected pair, render Spearman and Kendall versions using identical models and axes; colours and borders follow the metric named on that figure. Each panel annotates both coefficients and model counts for both sample scopes. Large selections are paginated using `PAIR_PANELS_PER_FIGURE`, with pages kept separate by run and metric. A run with no qualifying shared pairs is reported explicitly.

`PAIR_METRICS` controls which metrics are rendered; both are always calculated. `PAIR_SCOPE` chooses `competitive` or `all_final` for heatmaps and shared pair selection. The defaults require 20 distinct models and 3 distinct values per parameter. `PAIR_MIN_ABS_RHO` controls only the shared Spearman selection, not heatmap visibility or a Kendall threshold. Direct ConfigSpace relations are excluded from automatic scatter selection unless `PAIR_INCLUDE_CONSTRAINED=True`.

The analysed scope uses the correlation colour. In competitive scope, all final-stage models form a grey background. In all-final scope, the full sample is coloured and competitive models have black outlines. Both axes respect `LOG_PARAMETERS`.

These are exploratory rank associations, **not posterior correlations or significance tests**. Adaptive search, bounds, imposed inequalities and competitive-model selection can induce associations; choosing the largest coefficients among many pairs can exaggerate apparent strength. Nonmonotonic dependencies may be missed. The pair audit, metric-specific strength counts and figure index are included in optional CSV exports.

In [ ]:
# 1. Compute both metrics on the same model pairs, then show per-run heatmaps.
# PARAMETERS controls marginal plots; this audit includes all independently varied parameters.
if not PAIR_METRICS:
    raise ValueError('Select at least one display metric in PAIR_METRICS.')
for metric in PAIR_METRICS:
    correlation_metric(metric)  # Validate metric names before rendering.
pair_analysis = parameter_pair_analysis(
    plot_evaluations, plotted_competitive, configspaces, parameters=None,
    min_models=PAIR_MIN_MODELS, min_unique=PAIR_MIN_UNIQUE_VALUES,
    periodic_parameters=PAIR_PERIODIC_PARAMETERS)
for scope in ('competitive', 'all_final'):
    pair_analysis[f'level_{scope}'] = pair_analysis[f'rho_{scope}'].where(
        pair_analysis[f'eligible_{scope}'].eq(True)).map(correlation_strength)
    pair_analysis[f'level_tau_b_{scope}'] = pair_analysis[f'tau_b_{scope}'].where(
        pair_analysis[f'eligible_{scope}'].eq(True)).map(correlation_strength)
level_tables = []
for metric in ('spearman', 'kendall_tau_b'):
    level_column = f'level_{PAIR_SCOPE}' if metric == 'spearman' else f'level_tau_b_{PAIR_SCOPE}'
    summary = (pair_analysis.groupby(['history', 'objective_key', level_column], sort=False)
               .size().rename('parameter_pairs').reset_index()
               .rename(columns={level_column: 'correlation_level'}))
    summary['metric'] = metric
    level_tables.append(summary)
correlation_level_summary = pd.concat(level_tables, ignore_index=True)
pair_plot_rows = []
if pair_analysis.empty:
    print('No independently varied parameter pairs at the configured final stage.')
else:
    display(correlation_level_summary)
    display(pair_analysis.groupby(['history', f'reason_{PAIR_SCOPE}'], sort=False)
            .size().rename('parameter_pairs').reset_index())
    for group_index, ((history, objective), group) in enumerate(
            pair_analysis.groupby(['history', 'objective_key'], sort=False), start=1):
        print('Observation products:', group.products.iloc[0])
        for metric in PAIR_METRICS:
            _, metric_label, _ = correlation_metric(metric)
            display(Markdown(f'#### {metric_label} heatmap: {history}'))
            suffix = '' if metric == 'spearman' else '_kendall_tau_b'
            heatmap_name = f'parameter_correlation_heatmap_{group_index:02d}{suffix}'
            association_name = f'parameter_associations_{group_index:02d}{suffix}'
            pair_plot_rows.append({'history': history, 'objective_key': objective,
                                   'scope': PAIR_SCOPE, 'metric': metric,
                                   'selection_metric': 'spearman', 'heatmap_figure': heatmap_name,
                                   'association_figure': association_name, 'association_panels': 0})
            show_figure(heatmap_name, plot_parameter_correlation_heatmap(group, PAIR_SCOPE, metric=metric))
pair_plot_index = pd.DataFrame(pair_plot_rows, columns=[
    'history', 'objective_key', 'scope', 'metric', 'selection_metric',
    'heatmap_figure', 'association_figure', 'association_panels'])

In [ ]:
# 2. Keep one shared Spearman-selected pair list per run; show both metric versions.
selected_pair_groups = []
rank_columns = ['parameter_x', 'parameter_y', f'level_{PAIR_SCOPE}', f'level_tau_b_{PAIR_SCOPE}',
                'n_competitive', 'rho_competitive', 'tau_b_competitive',
                'n_all_final', 'rho_all_final', 'tau_b_all_final', 'direct_configspace_constraint']
for group_index, ((history, objective), group) in enumerate(
        pair_analysis.groupby(['history', 'objective_key'], sort=False), start=1):
    display(Markdown(f'#### Shared parameter associations: {history}'))
    selected_for_run = select_parameter_pairs(
        group, scope=PAIR_SCOPE, min_abs_rho=PAIR_MIN_ABS_RHO,
        max_plots=PAIR_MAX_PER_HISTORY, max_per_history=PAIR_MAX_PER_HISTORY,
        include_constrained=PAIR_INCLUDE_CONSTRAINED)
    if selected_for_run.empty:
        print('No pairs meet the shared sample-count, variation and Spearman thresholds for this run.')
        continue
    selected_for_run['selection_metric'] = 'spearman'
    selected_pair_groups.append(selected_for_run)
    display(selected_for_run[rank_columns])
    if PAIR_PANELS_PER_FIGURE < 1:
        raise ValueError('PAIR_PANELS_PER_FIGURE must be positive.')
    for metric in PAIR_METRICS:
        run_figure_names = []
        suffix = '' if metric == 'spearman' else '_kendall_tau_b'
        for first in range(0, len(selected_for_run), PAIR_PANELS_PER_FIGURE):
            page = selected_for_run.iloc[first:first+PAIR_PANELS_PER_FIGURE]
            name = f'parameter_associations_{group_index:02d}{suffix}'
            if len(selected_for_run) > PAIR_PANELS_PER_FIGURE:
                name += f'_page_{first//PAIR_PANELS_PER_FIGURE+1:02d}'
            run_figure_names.append(name)
            show_figure(name, plot_selected_parameter_pairs(
                page, plot_evaluations, plotted_competitive, LOG_PARAMETERS, metric=metric))
        mask = ((pair_plot_index.history == history) & (pair_plot_index.objective_key == objective)
                & (pair_plot_index.metric == metric))
        pair_plot_index.loc[mask, 'association_panels'] = len(selected_for_run)
        pair_plot_index.loc[mask, 'association_figure'] = ', '.join(run_figure_names)
selected_pairs = (pd.concat(selected_pair_groups, ignore_index=True) if selected_pair_groups else
                  select_parameter_pairs(pair_analysis.iloc[:0], scope=PAIR_SCOPE))

## 6. PDI correction state and prediction amplitudes

Read the correction flag actually saved with each PDI evaluation. A corrected diagnostic image does not establish that corrected images entered the loss. `None` means the historical record does not establish the correction state.

The polarised-light fraction, normalised radial shape and quadrant terms carry different information. SSIM is a morphology diagnostic, not an amplitude test or an objective contribution in the current code. The tables below retain fractions as fractions; multiply by 100 for percent.

In [ ]:
correction_columns = ['history', 'budget', *[f'pdi_{b}_corrected' for b in 'VIH'],
                      *[f'pdi_{b}_correction_radius_px' for b in 'VIH']]
pdi_evaluations = evaluations.loc[evaluations.usable & evaluations[PDI].notna().any(axis=1)]
if not pdi_evaluations.empty:
    correction_audit = pdi_evaluations[correction_columns].drop_duplicates()
    display(correction_audit)
else:
    correction_audit = pd.DataFrame()
    print('No saved PDI scores in these histories.')
pdi_predictions, alma_predictions = prediction_tables(plotted_best, records)
display(pdi_predictions)
display(alma_predictions)
show_figure('prediction_amplitudes', plot_amplitudes(pdi_predictions, alma_predictions))

## 7. Saved radial predictions

Solid curves are models and dashed curves are their associated observations. PDI profiles are normalised annular contributions, not surface brightness or independent amplitude measurements. ALMA profiles retain Jy/beam. Different apertures or radial binning must be considered before interpreting differences.

Only data already stored in `additional_info` are used. SED and visibility point predictions are generally not stored there; their original comparison figures can be viewed in the optional gallery below.

In [ ]:
profile_selection = plotted_best.head(PROFILE_MAX_MODELS)
show_figure('pdi_radial_profiles', plot_saved_profiles(profile_selection, records, 'pdi'))
show_figure('alma_radial_profiles', plot_saved_profiles(profile_selection, records, 'alma'))

## 8. Physical model files, resolved zone masses and input profiles

Configuration IDs are local to a history and may collide between optimiser seeds sharing a trials folder. Match the sampled parameters and saved fidelity against `config_used.json` before linking a trial folder. Symbolic ties are resolved during the match. Unmatched historical directory formats are reported, not guessed.

The profiles reconstruct the saved **input** surface-density law, normalising each zone separately to its saved dust mass. They are not recovered MCFOST density/temperature fields. The scale-height profile is the parameterised reference structure, before grain-dependent settling. Surface density uses solar masses and au converted to g/cm². No model assumptions or parameters are changed.

In [ ]:
resolved_models, model_rows = {}, []
for _, row in plotted_best.iterrows():
    matches = resolve_model_files(row, records)
    if matches:
        saved = matches[0]
        resolved_models[row.evaluation_id] = saved
        cfg = saved['effective_cfg']
        model_rows.append({'history': row.history, 'config_id': row.config_id,
                           'matching_trial_folders': len(matches), 'trial_folder': str(saved['folder']),
                           'model_parameter_file': str(saved['folder'] / 'model.para'),
                           'zone_1_dust_mass_Msun': cfg.get('zone_1_dust_mass'),
                           'zone_2_dust_mass_Msun': cfg.get('zone_2_dust_mass'),
                           'zone_1_Rout_au': cfg.get('zone_1_Rout'), 'zone_2_Rin_au': cfg.get('zone_2_Rin')})
    else:
        model_rows.append({'history': row.history, 'config_id': row.config_id, 'matching_trial_folders': 0})
model_files = pd.DataFrame(model_rows)
display(model_files)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
profile_count = 0
for _, row in plotted_best.iterrows():
    saved = resolved_models.get(row.evaluation_id)
    if not saved:
        continue
    cfg = saved['effective_cfg']
    color = f'C{profile_count % 10}'
    labelled = False
    for zone in (1, 2):
        prefix = f'zone_{zone}_'
        required = [prefix+k for k in ('Rin', 'Rout', 'dust_mass', 'surface_density_exp', 'scale_height', 'Rref', 'flaring_exp')]
        if not all(k in cfg for k in required):
            continue
        rin, rout = float(cfg[prefix+'Rin']), float(cfg[prefix+'Rout'])
        if not 0 < rin < rout or float(cfg.get(prefix+'type', 1)) not in (1, 2):
            continue
        radius = np.geomspace(rin, rout, 1000)
        shape = (radius/rin)**float(cfg[prefix+'surface_density_exp'])
        if float(cfg.get(prefix+'type', 1)) == 2:
            power = 2 + float(cfg[prefix+'-gamma_exp'])
            rc = float(cfg[prefix+'Rc'])
            shape *= np.exp(-(radius/rc)**power + (rin/rc)**power)
        area_integral = 2*np.pi*trapezoid(shape*radius, radius)
        sigma = float(cfg[prefix+'dust_mass']) / area_integral * shape * 1.98847e33/(1.495978707e13**2)
        height = float(cfg[prefix+'scale_height'])*(radius/float(cfg[prefix+'Rref']))**float(cfg[prefix+'flaring_exp'])
        label = row.history if not labelled else None
        axes[0].loglog(radius, sigma, color=color, label=label)
        axes[1].loglog(radius, height, color=color, label=label)
        labelled = True
    profile_count += int(labelled)
if profile_count:
    axes[0].set(xlabel='Radius [au]', ylabel='Input dust surface density [g/cm²]')
    axes[1].set(xlabel='Radius [au]', ylabel='Parameterised scale height [au]')
    axes[0].legend(fontsize=6)
    fig.tight_layout()
    show_figure('input_physical_profiles', fig)
else:
    plt.close(fig)
    print('No matched two-zone input profiles available.')

## 9. Add observations using a common model pool

Compare models that already have every recognised observation contribution. The pool is partitioned using known ConfigSpace/template fingerprints, observation descriptors, PDI correction state and inferred PDI term weights. Missing metadata or unknown historical code/data revisions still limit comparability. Inspect inventory and scoring definitions before drawing physical conclusions across folders.

Subset objectives are reconstructed from the **unchanged recorded contributions**. The same candidates are evaluated for each subset, separating observational preference from differences in search coverage. Missing observables are never assigned zero. Leave-one-group-out comparisons show which observations drive the selected compromise.

Exact copied evaluations are deduplicated within a compatible pool. Distinct stochastic scores remain, so winning evaluations are still subject to simulation noise.
Recorded numerical fidelity settings also partition pools. Missing historical photon/resolution metadata still prevents establishing numerical equivalence. Pool membership requires complete, matching cost bookkeeping.


In [ ]:
common_pool, subset_winners = common_pool_comparison(evaluations)
if common_pool.empty:
    print('No compatible pool with every observation contribution; this is expected for SED-only selections.')
    pool_groups = pd.DataFrame()
    selected_pool = common_pool
    selected_subset_winners = subset_winners
else:
    pool_groups = common_pool.groupby('pool_group').agg(
        evaluations=('evaluation_id', 'size'), models=('parameter_hash', 'nunique'), histories=('history', 'nunique'))
    display(pool_groups)
    selected_group = POOL_GROUP or pool_groups.evaluations.idxmax()
    print('Displaying compatibility group:', selected_group)
    selected_pool = common_pool[common_pool.pool_group == selected_group].copy()
    selected_subset_winners = subset_winners[subset_winners.pool_group == selected_group]
    display(selected_subset_winners)
    show_figure('common_pool_tradeoffs', plot_tradeoffs(selected_pool))

In [ ]:
observation_changes = pd.DataFrame()
if not selected_pool.empty:
    changes = []
    for old_name, new_name in zip(OBSERVATION_SEQUENCE, OBSERVATION_SEQUENCE[1:]):
        if old_name not in SUBSETS or new_name not in SUBSETS:
            raise ValueError('Unknown subset in OBSERVATION_SEQUENCE.')
        if not set(SUBSETS[old_name]).issubset(SUBSETS[new_name]):
            raise ValueError('Each observation addition must retain all previous objective terms.')
        old_scores = selected_pool[SUBSETS[old_name]].sum(axis=1)
        new_scores = selected_pool[SUBSETS[new_name]].sum(axis=1)
        old_id, new_id = old_scores.idxmin(), new_scores.idxmin()
        changes.append({'addition': f'{old_name} → {new_name}',
                        'old_model': selected_pool.loc[old_id, 'evaluation_id'],
                        'new_model': selected_pool.loc[new_id, 'evaluation_id'],
                        'old_objective_before': old_scores.loc[old_id],
                        'old_objective_after': old_scores.loc[new_id],
                        'old_objective_degradation': old_scores.loc[new_id]-old_scores.loc[old_id],
                        'new_objective_before': new_scores.loc[old_id],
                        'new_objective_after': new_scores.loc[new_id],
                        'new_objective_improvement': new_scores.loc[old_id]-new_scores.loc[new_id]})
    observation_changes = pd.DataFrame(changes)
    display(observation_changes)
    chosen_ids = selected_subset_winners.evaluation_id.unique()
    common_winner_parameters = selected_pool[selected_pool.evaluation_id.isin(chosen_ids)]
    display(common_winner_parameters[['history', 'config_id', 'cost', *parameter_columns]])

## 10. Optional original model/data diagnostic gallery

Enable `SHOW_SAVED_DIAGNOSTICS` to inspect the existing SED, visibility, PDI and ALMA comparison figures for matched models. The gallery prioritises models with the most scored observables, then the lowest recorded cost for display; this is not a ranking across different objectives. Multiple matched folders are equivalent in sampled parameters but may contain different stochastic realisations.

These are original diagnostic figures. Their plotting scales or display normalisations may differ; inspect the amplitude tables and numerical residuals as well.

In [ ]:
if SHOW_SAVED_DIAGNOSTICS:
    gallery = plotted_best.assign(_groups=plotted_best[list(SCORE_COLUMNS.values())].notna().sum(axis=1))
    gallery = gallery.sort_values(['_groups', 'cost'], ascending=[False, True]).head(GALLERY_MAX_MODELS)
    for _, row in gallery.iterrows():
        saved = resolved_models.get(row.evaluation_id)
        if not saved:
            continue
        folder = saved['folder']
        candidates = [folder/'SED_Akke_model1_MCFOST.png', folder/'SED_secondary_comparison.png',
                      folder/'figures/visibilities_GRAVITY 2.2.png',
                      folder/'figures/visibilities_MATISSE L 3.5.png',
                      folder/'figures/pdi_constraints_i.png', folder/'figures/alma_comparison.png']
        available = [p for p in candidates if p.is_file()]
        if not available:
            continue
        fig, axes = plt.subplots(int(np.ceil(len(available)/2)), 2,
                                 figsize=(16, 5*int(np.ceil(len(available)/2))), squeeze=False)
        for ax, path in zip(axes.flat, available):
            ax.imshow(plt.imread(path)); ax.set_title(path.name, fontsize=9); ax.axis('off')
        for ax in axes.flat[len(available):]:
            ax.axis('off')
        fig.suptitle(f'{row.history}; config {row.config_id}')
        fig.tight_layout()
        show_figure('gallery_'+str(row.config_id)+'_'+str(len(figures)), fig)
else:
    print('Gallery disabled. Set SHOW_SAVED_DIAGNOSTICS=True in the settings cell to display saved comparison figures.')

## 11. Export tables, provenance and figures

Set `SAVE_OUTPUTS=True` to export to `OUTPUT_DIR`. Exports do not change the simulation folders. `evaluations.csv` retains exclusions and all repeated evaluations; top/competitive tables select distinct configurations. `history_sha256` identifies the snapshot read, which is useful for active runs.

When interpreting results, report the observation contributions, prediction amplitudes, correction states, physical assumptions, numerical settings and optimiser variability alongside the best cost.

In [ ]:
tables = {
    'inventory': inventory, 'coverage': coverage, 'evaluations': evaluations,
    'score_audit': score_audit, 'score_audit_summary': score_audit_summary,
    'score_audit_issues': score_audit_issues,
    'best_models': best, 'top_models': top_models, 'competitive_models': competitive,
    'competitive_parameter_ranges': parameter_ranges, 'stage_rank_correlations': rank_correlations,
    'parameter_pair_analysis': pair_analysis, 'selected_parameter_pairs': selected_pairs,
    'pair_correlation_levels': correlation_level_summary, 'pair_plot_index': pair_plot_index,
    'correction_audit': correction_audit, 'pdi_predictions': pdi_predictions,
    'alma_predictions': alma_predictions, 'model_files': model_files,
    'common_pool_subset_winners': subset_winners, 'observation_changes': observation_changes,
}
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
    for name, figure in figures.items():
        figure.savefig(OUTPUT_DIR / f'{name}.png', dpi=180, bbox_inches='tight')
    settings = {'run_folders': {k: str(v) for k, v in RUN_FOLDERS.items()},
                'penalty_limit': PENALTY_LIMIT, 'top_n': TOP_N,
                'relative_cost_allowance': RELATIVE_COST_ALLOWANCE,
                'absolute_cost_allowance': ABSOLUTE_COST_ALLOWANCE,
                'observation_sequence': OBSERVATION_SEQUENCE,
                'pair_analysis': {'scope': PAIR_SCOPE, 'metrics': list(PAIR_METRICS),
                                  'scatter_selection_metric': 'spearman', 'kendall_variant': 'b',
                                  'min_models': PAIR_MIN_MODELS,
                                  'min_unique_values': PAIR_MIN_UNIQUE_VALUES, 'min_abs_rho': PAIR_MIN_ABS_RHO,
                                  'max_per_history': PAIR_MAX_PER_HISTORY,
                                  'panels_per_figure': PAIR_PANELS_PER_FIGURE,
                                  'strength_thresholds': [0.2, 0.4, 0.6, 0.8],
                                  'periodic_parameters': sorted(PAIR_PERIODIC_PARAMETERS),
                                  'include_constrained': PAIR_INCLUDE_CONSTRAINED},
                'pool_group': None if common_pool.empty else selected_group,
                'interpretation': 'Recorded optimisation evaluations, not posterior samples. No simulations launched.'}
    (OUTPUT_DIR / 'analysis_settings.json').write_text(json.dumps(settings, indent=2))
    print(f'Exported {len(tables)} tables and {len(figures)} figures to {OUTPUT_DIR}')
else:
    print('Export disabled. Tables and figures are available in memory; set SAVE_OUTPUTS=True to save them.')

## Reading the results critically

1. **Inventory and coverage:** each optimiser seed is a separate history. Usability filters execution failures, penalties, missing configurations and non-finite costs; it does not establish that every observational component was recorded correctly. Use the separate score audit for that check.
2. **Best and competitive models:** selection uses the configured final budget and ranks each observation product set separately. Keeping the lowest repeat for each configuration favours favourable noise realisations; these are best observed scores, not estimates of mean performance. With the present relative allowance of 1.0 and positive costs, competitive means cost at most twice the best cost. TOP_N limits the top table only.
3. **Convergence and stage ranks:** a change in the observation set changes the objective. Rank comparisons use only promoted/repeated configurations observed at both budgets and therefore have selection bias. Their agreement does not establish convergence over the full parameter space.
4. **Parameters and correlations:** competitive ranges are sampled ranges, not uncertainty intervals. The current all_final scope includes poor models and optimiser sampling preferences. Strong Spearman/Kendall associations may reflect adaptive proposals, imposed constraints or cost selection; weak associations do not rule out nonlinear degeneracy. Numeric categorical values can also be ranked by this implementation: only interpret their order when physically meaningful. PA is excluded from ordinary rank correlations.
5. **PDI and ALMA predictions:** inspect amplitudes together with morphology. PDI radial profiles are normalised and do not independently establish absolute brightness. Internal PDI weights remain embedded in the saved losses; adding them again would double-count weighting. SSIM is diagnostic only in the current objective.
6. **Model files and physical profiles:** matching parameters and fidelity is safer than matching local configuration IDs, but multiple stochastic realisations may still match. Reconstructed profiles show the input density/scale-height prescription, not the simulated settled density or temperature fields. The first matched folder need not be the exact stochastic evaluation that produced the winning score.
7. **Common-pool observation additions:** every candidate must have all nine finite contributions and pass cost bookkeeping. Known physical, scoring and numerical metadata partition the pool. The largest pool is chosen by default, which can hide other groups; inspect the group table. Unknown historical code/data revisions or missing numerical metadata prevent proving equivalence. This pool need not represent the full physically allowed parameter space, and subset winners are conditional on its original search coverage. No new optimisation or posterior inference is performed.
8. **Exports:** all evaluation rows, audit statuses and provenance are retained. Tables and plots can be saved with SAVE_OUTPUTS; raw observations, uncertainties, weights and simulation outputs are unchanged.

**Snapshot validation:** the selected Dubrulle collection contained 8,002 recorded evaluations in six histories when checked during this revision: 7,298 usable evaluations all had complete component sums matching cost at rtol=atol=1e-8; 704 excluded evaluations had incomplete score records. Active histories can change, so the executable audit above is authoritative for each rerun. The largest absolute roundoff difference was 0.0001220703125; agreement is assessed relative to cost as well as with an absolute tolerance, not by exact float equality.
